# Chapter 2 | Predictive Modelling of Student Academic Performance

**Reproducible companion notebook**  
*Decision Tree classification · Mathematics course · Portuguese secondary-school data*

This notebook follows Chapter 2 of *Artificial Intelligence in Education*. It covers dataset preparation, exploratory analysis, a stratified training/test split, ten-fold cross-validation, model evaluation, feature interpretation, and educational implications. Charts are aligned with the chapter's data-based figures (Figures 2.2–2.12); Figure 2.1 is a conceptual framework rather than a computational result.

**Dataset:** [Student Performance, UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/320/student+performance)  
**Citation:** Cortez, P. (2008). *Student Performance* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5TG7T  
**Dataset licence:** CC BY 4.0. The Mathematics file (`student-mat.csv`) is used; the Portuguese-language course file is not included in this analysis.

**Execution:** Run cells sequentially. Python 3.10+ with `pandas`, `numpy`, `matplotlib`, `seaborn`, and `scikit-learn` is required. If the Mathematics CSV is not found locally, the notebook attempts to download it from the official UCI archive. An internet connection is required only for that initial download. Generated charts are saved under `chapter2_outputs/`.

## 1. Environment and data preparation

<div style="border-left: 4px solid #46657a; background: #f4f7f9; padding: 10px 14px; border-radius: 6px; margin: 14px 0 7px;">
<strong style="color: #263f52;">Libraries and reproducibility</strong><br>
<span style="color: #4b5964; font-size: 0.94em;">Import the tools used throughout the chapter and define a consistent chart style.</span>
</div>

In [ ]:
from pathlib import Path
from io import BytesIO
from zipfile import ZipFile, is_zipfile
from urllib.request import Request, urlopen
import hashlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, roc_auc_score, roc_curve,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42
OUTPUT_DIR = Path('chapter2_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SLATE = '#4b6475'
SLATE_LIGHT = '#9eb5c5'
ROSEWOOD = '#9a4f60'
TEXT = '#243746'
sns.set_theme(style='white', context='notebook')
plt.rcParams.update({
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'savefig.bbox': 'tight',
    'figure.dpi': 115,
})


def save_figure(fig, stem):
    """Save an editable vector figure and a high-resolution preview."""
    fig.savefig(OUTPUT_DIR / f'{stem}.pdf', bbox_inches='tight')
    fig.savefig(OUTPUT_DIR / f'{stem}.png', dpi=220, bbox_inches='tight')
    plt.show()
    plt.close(fig)

<div style="border-left: 4px solid #46657a; background: #f4f7f9; padding: 10px 14px; border-radius: 6px; margin: 14px 0 7px;">
<strong style="color: #263f52;">Load the official Mathematics dataset</strong><br>
<span style="color: #4b5964; font-size: 0.94em;">Read a local copy when available; otherwise retrieve and cache the CSV from UCI.</span>
</div>

In [ ]:
LOCAL_CANDIDATES = [Path('student-mat.csv'), Path('data/student-mat.csv'),
                    Path('datasets/student-mat.csv')]
UCI_ARCHIVE = 'https://archive.ics.uci.edu/static/public/320/student%2Bperformance.zip'


def mathematics_csv_from_archive(archive_bytes):
    """Locate student-mat.csv, including archives that contain another ZIP file."""
    with ZipFile(BytesIO(archive_bytes)) as archive:
        for member in archive.namelist():
            if member.endswith('/'):
                continue
            if Path(member).name.lower() == 'student-mat.csv':
                return archive.read(member)
        for member in archive.namelist():
            if member.lower().endswith('.zip'):
                nested = archive.read(member)
                if is_zipfile(BytesIO(nested)):
                    try:
                        return mathematics_csv_from_archive(nested)
                    except FileNotFoundError:
                        pass
    raise FileNotFoundError('student-mat.csv was not found in the UCI archive.')


source_path = next((p for p in LOCAL_CANDIDATES if p.is_file()), None)
if source_path is None:
    try:
        request = Request(UCI_ARCHIVE, headers={'User-Agent': 'Mozilla/5.0'})
        with urlopen(request, timeout=35) as response:
            data_bytes = mathematics_csv_from_archive(response.read())
        source_path = Path('data/student-mat.csv')
        source_path.parent.mkdir(parents=True, exist_ok=True)
        source_path.write_bytes(data_bytes)
    except Exception as exc:
        raise FileNotFoundError(
            'The dataset could not be downloaded. Download the original Student '
            'Performance archive from https://archive.ics.uci.edu/dataset/320/'
            'student+performance, extract student-mat.csv, and place it beside '
            'this notebook or in the data/ folder.'
        ) from exc

raw_csv = source_path.read_bytes()
df = pd.read_csv(BytesIO(raw_csv), sep=';')
print(f'Dataset: {source_path} | shape: {df.shape[0]} rows × {df.shape[1]} columns')
print(f'SHA-256: {hashlib.sha256(raw_csv).hexdigest()}')

<div style="border-left: 4px solid #46657a; background: #f4f7f9; padding: 10px 14px; border-radius: 6px; margin: 14px 0 7px;">
<strong style="color: #263f52;">Validate the data and define the outcome</strong><br>
<span style="color: #4b5964; font-size: 0.94em;">Confirm the expected Mathematics dataset structure and classify final grades at the threshold of 10.</span>
</div>

In [ ]:
required_columns = {'G1', 'G2', 'G3', 'studytime', 'absences'}
missing_columns = required_columns - set(df.columns)
assert not missing_columns, f'Missing columns: {sorted(missing_columns)}'
assert df.shape == (395, 33), (
    'The analysis expects the 395-row Mathematics dataset with 33 columns.'
)

for grade in ['G1', 'G2', 'G3']:
    df[grade] = pd.to_numeric(df[grade], errors='raise')

assert df[['G1', 'G2', 'G3']].notna().all().all(), 'Grade values must be present.'
assert df[['G1', 'G2', 'G3']].apply(lambda col: col.between(0, 20)).all().all()
df['pass'] = (df['G3'] >= 10).astype(int)

print('Outcome: Fail = 0 (G3 < 10), Pass = 1 (G3 >= 10)')
print(df['pass'].value_counts().sort_index().rename(index={0: 'Fail', 1: 'Pass'}))
print(f'Missing entries: {int(df.isna().sum().sum())}')
display(df.head())

## 2. Exploratory analysis

<div style="border-left: 4px solid #46657a; background: #f4f7f9; padding: 10px 14px; border-radius: 6px; margin: 14px 0 7px;">
<strong style="color: #263f52;">Figure 2.2 · Pass and fail distribution</strong><br>
<span style="color: #4b5964; font-size: 0.94em;">Display the number of learners in each final-grade category.</span>
</div>

In [ ]:
counts = df['pass'].value_counts().reindex([0, 1], fill_value=0)
fig, ax = plt.subplots(figsize=(7.8, 4.7))
bars = ax.bar(['Fail (0)', 'Pass (1)'], counts, color=[SLATE, ROSEWOOD], width=0.57)
for bar in bars:
    ax.annotate(f'{bar.get_height():.0f}',
                (bar.get_x() + bar.get_width()/2, bar.get_height()),
                ha='center', va='bottom', xytext=(0, 5),
                textcoords='offset points')
ax.set(title='Distribution of Pass and Fail Outcomes', ylabel='Number of Students',
       ylim=(0, max(counts) * 1.14))
sns.despine(ax=ax)
fig.tight_layout()
save_figure(fig, 'Fig_2_2_Pass_Fail_Distribution')

<div style="border-left: 4px solid #46657a; background: #f4f7f9; padding: 10px 14px; border-radius: 6px; margin: 14px 0 7px;">
<strong style="color: #263f52;">Figure 2.3 · Final grade distribution</strong><br>
<span style="color: #4b5964; font-size: 0.94em;">Inspect the range and frequency of final Mathematics grades before binarisation.</span>
</div>

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.8))
sns.histplot(df['G3'], bins=np.arange(-0.5, 21.5, 1), kde=True,
             color=SLATE, edgecolor='white', ax=ax)
ax.set(title='Distribution of Final Grade (G3)', xlabel='Final Grade (0–20)',
       ylabel='Number of Students', xlim=(-0.5, 20.5))
sns.despine(ax=ax)
fig.tight_layout()
save_figure(fig, 'Fig_2_3_Final_Grade_Distribution')

<div style="border-left: 4px solid #46657a; background: #f4f7f9; padding: 10px 14px; border-radius: 6px; margin: 14px 0 7px;">
<strong style="color: #263f52;">Figure 2.4 · Correlation structure</strong><br>
<span style="color: #4b5964; font-size: 0.94em;">Show pairwise associations among numerical variables; correlations do not establish causal relationships.</span>
</div>

In [ ]:
correlations = df.select_dtypes(include=np.number).corr()
mask = np.triu(np.ones(correlations.shape, dtype=bool))
fig, ax = plt.subplots(figsize=(10.5, 8.7))
sns.heatmap(correlations, mask=mask,
            cmap=sns.diverging_palette(230, 15, as_cmap=True),
            vmin=-1, vmax=1, center=0, square=True, linewidths=0.4,
            cbar_kws={'shrink': 0.7, 'label': 'Pearson correlation'}, ax=ax)
ax.set_title('Correlation Matrix of Numerical Variables', pad=13)
ax.tick_params(axis='x', labelrotation=45)
ax.tick_params(axis='y', labelrotation=0)
fig.tight_layout()
save_figure(fig, 'Fig_2_4_Correlation_Matrix')

<div style="border-left: 4px solid #46657a; background: #f4f7f9; padding: 10px 14px; border-radius: 6px; margin: 14px 0 7px;">
<strong style="color: #263f52;">Figure 2.5 · Study time and final grade</strong><br>
<span style="color: #4b5964; font-size: 0.94em;">Compare grade distributions across the four reported weekly study-time categories.</span>
</div>

In [ ]:
study_labels = {
    1: '<2 h', 2: '2–5 h', 3: '5–10 h', 4: '>10 h'
}
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=df, x='studytime', y='G3', order=[1, 2, 3, 4],
            color=SLATE_LIGHT, width=0.55, fliersize=3, ax=ax)
ax.set(title='Weekly Study Time and Final Grade',
       xlabel='Weekly Study Time', ylabel='Final Grade (G3)')
ax.set_xticks(range(4), [study_labels[k] for k in [1, 2, 3, 4]])
sns.despine(ax=ax)
fig.tight_layout()
save_figure(fig, 'Fig_2_5_Study_Time_vs_Grade')

<div style="border-left: 4px solid #46657a; background: #f4f7f9; padding: 10px 14px; border-radius: 6px; margin: 14px 0 7px;">
<strong style="color: #263f52;">Figure 2.6 · Attendance and final grade</strong><br>
<span style="color: #4b5964; font-size: 0.94em;">Visualise absenteeism alongside grades; the fitted line summarises association only.</span>
</div>

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.9))
sns.regplot(data=df, x='absences', y='G3',
            scatter_kws={'color': SLATE, 'alpha': 0.35, 's': 22},
            line_kws={'color': ROSEWOOD, 'linewidth': 1.9},
            ci=95, ax=ax)
ax.set(title='Absences and Final Grade',
       xlabel='Number of Absences', ylabel='Final Grade (G3)')
sns.despine(ax=ax)
fig.tight_layout()
save_figure(fig, 'Fig_2_6_Absences_vs_Grade')

## 3. Predictive modelling

The final grade (`G3`) is used **only to define the target**; it is excluded from the predictors. Earlier grades (`G1`, `G2`) remain as input variables, as in the chapter. This makes the analysis relevant to **short-horizon support**, not to prediction at the very start of a course.

<div style="border-left: 4px solid #46657a; background: #f4f7f9; padding: 10px 14px; border-radius: 6px; margin: 14px 0 7px;">
<strong style="color: #263f52;">Predictors and stratified partitioning</strong><br>
<span style="color: #4b5964; font-size: 0.94em;">Separate input fields from the outcome and reserve 30% of cases for independent testing.</span>
</div>

In [ ]:
X = df.drop(columns=['G3', 'pass'])
y = df['pass']
categorical_columns = X.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_columns = X.select_dtypes(include=np.number).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)
print(f'Training cases: {len(X_train)} | Test cases: {len(X_test)}')
print(f'Categorical predictors: {len(categorical_columns)} | Numerical predictors: {len(numerical_columns)}')
print('Training class distribution:')
display(y_train.value_counts().sort_index().rename(index={0:'Fail',1:'Pass'}).to_frame('Count'))

<div style="border-left: 4px solid #46657a; background: #f4f7f9; padding: 10px 14px; border-radius: 6px; margin: 14px 0 7px;">
<strong style="color: #263f52;">Decision Tree pipeline</strong><br>
<span style="color: #4b5964; font-size: 0.94em;">Encode categorical predictors and fit a depth-limited classifier without scaling the numerical variables.</span>
</div>

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_columns),
        ('num', 'passthrough', numerical_columns),
    ]
)
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE)),
])
print('Classifier: Decision Tree | maximum depth: 5 | random state: 42')

<div style="border-left: 4px solid #46657a; background: #f4f7f9; padding: 10px 14px; border-radius: 6px; margin: 14px 0 7px;">
<strong style="color: #263f52;">Ten-fold cross-validation on training data</strong><br>
<span style="color: #4b5964; font-size: 0.94em;">Evaluate training-set variability without consulting the held-out test cases.</span>
</div>

In [ ]:
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)
cv_results = cross_validate(
    model, X_train, y_train, cv=cv,
    scoring={'accuracy': 'accuracy', 'f1_pass': 'f1', 'roc_auc': 'roc_auc'},
    n_jobs=1, error_score='raise',
)
cv_summary = pd.DataFrame({
    'Metric': ['Accuracy', 'F1 (Pass = 1)', 'ROC-AUC'],
    'Mean': [cv_results['test_accuracy'].mean(),
             cv_results['test_f1_pass'].mean(),
             cv_results['test_roc_auc'].mean()],
    'SD': [cv_results['test_accuracy'].std(ddof=1),
           cv_results['test_f1_pass'].std(ddof=1),
           cv_results['test_roc_auc'].std(ddof=1)],
})
display(cv_summary.round(3))

<div style="border-left: 4px solid #46657a; background: #f4f7f9; padding: 10px 14px; border-radius: 6px; margin: 14px 0 7px;">
<strong style="color: #263f52;">Fit the final model and evaluate the test set</strong><br>
<span style="color: #4b5964; font-size: 0.94em;">Train on the complete training partition and calculate all reported measures from predictions.</span>
</div>

In [ ]:
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_score = model.predict_proba(X_test)[:, list(model.classes_).index(1)]
report = classification_report(y_test, y_pred, labels=[0, 1],
                               target_names=['Fail', 'Pass'], output_dict=True,
                               zero_division=0)
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
accuracy = accuracy_score(y_test, y_pred)
f1_pass = f1_score(y_test, y_pred, pos_label=1)
roc_auc = roc_auc_score(y_test, y_score)

print(f'Accuracy: {accuracy:.4f} | F1 (Pass): {f1_pass:.4f} | ROC-AUC: {roc_auc:.4f}')
print('Confusion matrix (rows: actual; columns: predicted):')
print(cm)
print('\nClassification report:')
print(classification_report(y_test, y_pred, labels=[0, 1],
                            target_names=['Fail', 'Pass'], zero_division=0))

<div style="border-left: 4px solid #46657a; background: #f4f7f9; padding: 10px 14px; border-radius: 6px; margin: 14px 0 7px;">
<strong style="color: #263f52;">Reference comparison with Chapter 2</strong><br>
<span style="color: #4b5964; font-size: 0.94em;">Compare computed values with those printed in the book; reference values are never substituted for results.</span>
</div>

In [ ]:
chapter_reference = {
    'Accuracy': 0.8823529411764706,
    'F1 (Pass = 1)': 0.9078947368421053,
    'Fail support': 39,
    'Pass support': 80,
}
observed = {
    'Accuracy': accuracy,
    'F1 (Pass = 1)': f1_pass,
    'Fail support': int((y_test == 0).sum()),
    'Pass support': int((y_test == 1).sum()),
}
comparison = pd.DataFrame({
    'Metric': list(chapter_reference),
    'Book reference': list(chapter_reference.values()),
    'Recomputed': [observed[key] for key in chapter_reference],
})
comparison['Difference'] = comparison['Recomputed'] - comparison['Book reference']
display(comparison.round(5))
reference_cm = np.array([[36, 3], [11, 69]])
print('Confusion matrix matches the book:', np.array_equal(cm, reference_cm))
if not np.array_equal(cm, reference_cm):
    print('Review dataset, package versions and modelling settings before citing reproduced values.')

## 4. Model evaluation and visual analytics

<div style="border-left: 4px solid #46657a; background: #f4f7f9; padding: 10px 14px; border-radius: 6px; margin: 14px 0 7px;">
<strong style="color: #263f52;">Figure 2.7 · Overall classification performance</strong><br>
<span style="color: #4b5964; font-size: 0.94em;">Summarise accuracy, positive-class F1, and class-averaged F1 measures computed from the test set.</span>
</div>

In [ ]:
overall_metrics = pd.Series({
    'Accuracy': accuracy,
    'F1 (Pass)': f1_pass,
    'Macro F1': report['macro avg']['f1-score'],
    'Weighted F1': report['weighted avg']['f1-score'],
})
fig, ax = plt.subplots(figsize=(8.4, 4.9))
bars = ax.bar(overall_metrics.index, overall_metrics.values,
              color=[SLATE, '#668397', ROSEWOOD, '#b17a83'], width=0.59)
for bar, value in zip(bars, overall_metrics.values):
    ax.annotate(f'{value:.2f}',
                (bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 5), textcoords='offset points', ha='center')
ax.set(title='Overall Classification Performance', ylabel='Score', ylim=(0, 1.07))
sns.despine(ax=ax)
fig.tight_layout()
save_figure(fig, 'Fig_2_7_Overall_Performance')

<div style="border-left: 4px solid #46657a; background: #f4f7f9; padding: 10px 14px; border-radius: 6px; margin: 14px 0 7px;">
<strong style="color: #263f52;">Figure 2.8 · Class-wise performance</strong><br>
<span style="color: #4b5964; font-size: 0.94em;">Compare precision, recall and F1 for the fail and pass groups without hard-coded scores.</span>
</div>

In [ ]:
class_labels = ['Fail (0)', 'Pass (1)']
metric_names = ['Precision', 'Recall', 'F1-score']
keys = ['precision', 'recall', 'f1-score']
values = np.array([[report[label][key] for key in keys]
                   for label in ['Fail', 'Pass']])
positions = np.arange(2)
width = 0.23
fig, ax = plt.subplots(figsize=(8.5, 5.1))
for i, (name, color) in enumerate(zip(metric_names,
                                      [SLATE, SLATE_LIGHT, ROSEWOOD])):
    bars = ax.bar(positions + (i-1)*width, values[:, i], width=width,
                  label=name, color=color)
    for bar, score in zip(bars, values[:, i]):
        ax.annotate(f'{score:.2f}',
                    (bar.get_x()+bar.get_width()/2, bar.get_height()),
                    xytext=(0, 4), textcoords='offset points',
                    ha='center', fontsize=9)
ax.set_xticks(positions, class_labels)
ax.set(title='Class-wise Performance Metrics', ylabel='Score', ylim=(0, 1.12))
ax.legend(loc='upper center', bbox_to_anchor=(0.5, 1.02),
          ncol=3, frameon=False)
sns.despine(ax=ax)
fig.tight_layout()
save_figure(fig, 'Fig_2_8_Classwise_Performance')

<div style="border-left: 4px solid #46657a; background: #f4f7f9; padding: 10px 14px; border-radius: 6px; margin: 14px 0 7px;">
<strong style="color: #263f52;">Figure 2.9 · Class support and F1</strong><br>
<span style="color: #4b5964; font-size: 0.94em;">Present the test-set class sizes alongside the corresponding F1 values.</span>
</div>

In [ ]:
supports = [report[label]['support'] for label in ['Fail', 'Pass']]
f1_classes = [report[label]['f1-score'] for label in ['Fail', 'Pass']]
fig, ax1 = plt.subplots(figsize=(7.9, 5))
bar_positions = np.arange(2)
ax1.bar(bar_positions, supports, color=[SLATE_LIGHT, SLATE], width=0.54)
for index, count in enumerate(supports):
    ax1.annotate(f'{int(count)}', (index, count), xytext=(0, 5),
                 textcoords='offset points', ha='center')
ax1.set_xticks(bar_positions, ['Fail (0)', 'Pass (1)'])
ax1.set(ylabel='Number of Test Cases',
        title='Class Support and F1-score', ylim=(0, max(supports)*1.2))
ax2 = ax1.twinx()
ax2.plot(bar_positions, f1_classes, color=ROSEWOOD, marker='o',
         linewidth=1.7, markersize=7, label='F1-score')
for index, value in enumerate(f1_classes):
    ax2.annotate(f'{value:.2f}', (index, value), xytext=(9, -12),
                 textcoords='offset points', color=ROSEWOOD)
ax2.set(ylabel='F1-score', ylim=(0, 1.08))
ax1.spines['top'].set_visible(False)
ax2.spines['top'].set_visible(False)
fig.tight_layout()
save_figure(fig, 'Fig_2_9_Class_Support_and_F1')

<div style="border-left: 4px solid #46657a; background: #f4f7f9; padding: 10px 14px; border-radius: 6px; margin: 14px 0 7px;">
<strong style="color: #263f52;">Figure 2.10 · Confusion matrix</strong><br>
<span style="color: #4b5964; font-size: 0.94em;">Locate correct and incorrect predictions, noting the different educational implications of each error type.</span>
</div>

In [ ]:
fig, ax = plt.subplots(figsize=(6.8, 5.4))
sns.heatmap(cm, annot=True, fmt='d', cmap=sns.light_palette(SLATE, as_cmap=True),
            xticklabels=['Fail', 'Pass'], yticklabels=['Fail', 'Pass'],
            linewidths=1, linecolor='white', square=True,
            cbar_kws={'label': 'Number of Students'}, ax=ax)
ax.set(title='Confusion Matrix', xlabel='Predicted Outcome', ylabel='Actual Outcome')
fig.tight_layout()
save_figure(fig, 'Fig_2_10_Confusion_Matrix')

print(f'Predicted Pass, actual Fail (missed support opportunity): {cm[0, 1]}')
print(f'Predicted Fail, actual Pass (unnecessary concern): {cm[1, 0]}')

<div style="border-left: 4px solid #46657a; background: #f4f7f9; padding: 10px 14px; border-radius: 6px; margin: 14px 0 7px;">
<strong style="color: #263f52;">Figure 2.11 · ROC curve</strong><br>
<span style="color: #4b5964; font-size: 0.94em;">Assess discrimination over alternative decision thresholds using Pass (1) as the positive class.</span>
</div>

In [ ]:
false_positive_rate, true_positive_rate, thresholds = roc_curve(y_test, y_score,
                                                                  pos_label=1)
fig, ax = plt.subplots(figsize=(7.1, 5.3))
ax.plot(false_positive_rate, true_positive_rate, color=ROSEWOOD,
        linewidth=2.2, label=f'ROC-AUC = {roc_auc:.3f}')
ax.plot([0, 1], [0, 1], '--', color='#7c8790', linewidth=1.2, label='Chance')
ax.set(xlim=(0, 1), ylim=(0, 1.02),
       title='ROC Curve', xlabel='False Positive Rate', ylabel='True Positive Rate')
ax.legend(loc='lower right', frameon=False)
sns.despine(ax=ax)
fig.tight_layout()
save_figure(fig, 'Fig_2_11_ROC_Curve')

<div style="border-left: 4px solid #46657a; background: #f4f7f9; padding: 10px 14px; border-radius: 6px; margin: 14px 0 7px;">
<strong style="color: #263f52;">Figure 2.12 · Decision Tree feature importance</strong><br>
<span style="color: #4b5964; font-size: 0.94em;">Rank encoded predictors by impurity-based importance; importance measures association, not causation.</span>
</div>

In [ ]:
raw_feature_names = model.named_steps['preprocessor'].get_feature_names_out()
importances = model.named_steps['classifier'].feature_importances_
feature_labels = {
    'G1': 'First Period Grade (G1)',
    'G2': 'Second Period Grade (G2)',
    'age': 'Student Age',
    'Medu': "Mother's Education Level",
    'Fedu': "Father's Education Level",
    'traveltime': 'Travel Time to School',
    'studytime': 'Weekly Study Time',
    'failures': 'Past Class Failures',
    'famrel': 'Family Relationship Quality',
    'freetime': 'Free Time After School',
    'goout': 'Going Out with Friends',
    'Dalc': 'Workday Alcohol Consumption',
    'Walc': 'Weekend Alcohol Consumption',
    'health': 'Current Health Status',
    'absences': 'Number of Absences',
    'famsize': 'Family Size',
    'Mjob': "Mother's Occupation",
    'Fjob': "Father's Occupation",
    'reason': 'Reason for Choosing School',
    'paid': 'Extra Paid Classes',
    'schoolsup': 'Extra Educational Support',
    'famsup': 'Family Educational Support',
    'activities': 'Extra-curricular Activities',
    'Pstatus': 'Parental Cohabitation Status',
    'guardian': 'Student Guardian',
}


def readable_feature(name):
    prefix, token = name.split('__', 1)
    if prefix == 'num':
        return feature_labels.get(token, token)
    for column in sorted(categorical_columns, key=len, reverse=True):
        if token.startswith(column + '_'):
            category = token[len(column)+1:].replace('_', ' ').title()
            return f'{feature_labels.get(column, column)} ({category})'
    return token.replace('_', ' ').title()


importance_df = pd.DataFrame({
    'Feature': [readable_feature(name) for name in raw_feature_names],
    'Importance': importances,
}).sort_values('Importance', ascending=False, kind='stable')

# Encoded levels are retained individually to reproduce the chapter's chart.
top_importance = importance_df.head(15).iloc[::-1]
fig, ax = plt.subplots(figsize=(9, 6.5))
ax.barh(top_importance['Feature'], top_importance['Importance'], color=SLATE)
ax.set(title='Top 15 Predictive Factors for Academic Performance',
       xlabel='Gini Importance Score', ylabel='')
sns.despine(ax=ax)
fig.tight_layout()
save_figure(fig, 'Fig_2_12_Feature_Importance')
display(importance_df.head(15).reset_index(drop=True).round(4))

## 5. Educational interpretation

The model's prediction is a **decision-support signal**, not a judgement of a learner's ability or a substitute for professional assessment. The inclusion of second-period grades (`G2`) makes the reported performance most relevant to later-stage interventions. Predictions should be interpreted alongside contextual information, checked for differential errors across student groups, and used only to inform proportionate support.

**Reproducibility notes:** The test partition is never used during cross-validation. Figures 2.7–2.12 are computed from the fitted classifier, rather than entered as fixed scores. If the results differ from those printed in Chapter 2, verify the original Mathematics CSV, software versions, and documented split and model settings. Source data remain credited to the UCI repository under CC BY 4.0.